In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/latest.pt
/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/log.jsonl
/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt
/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/latest.pt
/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/log.jsonl
/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/acts_train.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/acts_val.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/sc_train.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/sc_val.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/test.json
/kaggle/input/datasets/

In [2]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

# Read token securely from Kaggle Secrets
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

REPO_ID = "Exploration-Lab/CS-781-Capstone"
LOCAL_DIR = "/kaggle/working/Phase_4_CJPE"

# Download only Phase 4 files
path = snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    token=hf_token,
    allow_patterns=[
        "Phase_4_CJPE/**"
    ],
    local_dir=LOCAL_DIR,
)

print("Download completed!")
print("Dataset location:", path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Download completed!
Dataset location: /kaggle/working/Phase_4_CJPE


In [3]:
# ==============================================================================
# INSPECT ARCH2 CHECKPOINT STRUCTURE & ALL MOUNTED PHASE 1 MODELS
# ==============================================================================
import os
import torch

print("=== 1. ALL MOUNTED MODEL CHECKPOINTS IN /kaggle/input ===")
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith((".pt", ".pth", ".bin", ".json")):
            p = os.path.join(root, f)
            print(f"  [{os.path.getsize(p)/(1024*1024):7.2f} MB] {p}")

arch2_path = "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/latest.pt"
ckpt2 = torch.load(arch2_path, map_location="cpu", weights_only=False)

print("\n=== 2. ARCH2 CHECKPOINT TOP-LEVEL KEYS ===")
print("Keys:", list(ckpt2.keys()))
if "cfg" in ckpt2:
    print("Arch2 cfg:", ckpt2["cfg"])
elif "config" in ckpt2:
    print("Arch2 config:", ckpt2["config"])

state_key = "model" if "model" in ckpt2 else ("state_dict" if "state_dict" in ckpt2 else None)
sd = ckpt2[state_key] if state_key else ckpt2
print(f"\n=== 3. ARCH2 STATE_DICT SAMPLE KEYS (Total tensors: {len(sd)}) ===")
for k, v in list(sd.items())[:25]:
    if hasattr(v, "shape"):
        print(f"  {k:45s}: {tuple(v.shape)}")

=== 1. ALL MOUNTED MODEL CHECKPOINTS IN /kaggle/input ===
  [1041.64 MB] /kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/latest.pt
  [1041.62 MB] /kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt
  [1487.54 MB] /kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/latest.pt
  [1487.50 MB] /kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt
  [ 277.62 MB] /kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/acts_train.bin
  [   6.42 MB] /kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/acts_val.bin
  [ 476.42 MB] /kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/sc_train.bin
  [   9.15 MB] /kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/sc_val.bin
  [   2.89 MB] /kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/test.json
  [   2.74 MB] /kaggle/input

In [4]:
# ==============================================================================
# PHASE 4 (CJPE) — EXPERIMENT 3: FULL-CORPUS PRODUCTION PIPELINE
# ==============================================================================
#
# PRIMARY OBJECTIVE:
#   Predict binary court judgment outcomes (0 = Rejected/Dismissed, 1 = Accepted/Allowed)
#   across long Indian legal case documents (median ~2,200 words, max ~412k words),
#   evaluated strictly via unweighted Macro-F1.
#
# MANDATORY COMPETITION COMPLIANCE (SECTIONS 10.1 & 10.2):
#   1. Backbone Model: Phase 1 Arch2 Pre-LN Transformer (768-dim, 6 layers, 8 heads).
#   2. Adaptation Technique: BitFit [Zaken et al., 2022] + Layer-Wise Partial Unfreezing.
#   3. Long-Document Strategy: 4-Zone Hierarchical Character-Slicing with multi-layer pooling.
#   4. Training Protocol: 100% of labeled data, 3 seeds, leak-free 5-fold OOF lexical features.
#   5. Output Format: [{"id": ..., "pred": 0/1}, ...] inside submission.zip.
# ==============================================================================

import os
import re
import copy
import json
import math
import random
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import Counter
from tokenizers import Tokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.model_selection import StratifiedKFold
from scipy.special import expit, softmax
from tqdm.auto import tqdm

# ------------------------------------------------------------------------------
# STEP 0: Reproducibility & Device Configuration
# ------------------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_ROOT = Path("/kaggle/working/Phase_4_CJPE/Phase_4_CJPE")
BASE_OUT  = Path("/kaggle/working/Phase4_Submissions")
EXP3_DIR  = BASE_OUT / "Exp3_FullTrain_Arch2_BitFit_Unfreeze"
EXP3_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------------------------
# STEP 1: Load Full Labeled Corpus (Train + Dev = 39,375 Cases) & Test Set
# ------------------------------------------------------------------------------
with open(DATA_ROOT / "Train/train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(DATA_ROOT / "Train/dev.json", "r", encoding="utf-8") as f:
    dev_data = json.load(f)
with open(DATA_ROOT / "Test/test.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

all_records = train_data + dev_data
y_full = np.array([int(x["label"]) for x in all_records], dtype=np.int64)
print(f"[Step 1] Full Training Corpus: {len(all_records)} cases | Label Counts: {Counter(y_full)}")


# ------------------------------------------------------------------------------
# STEP 2: Legal Text De-Corruption & 4-Zone Long-Document Slicer
# ------------------------------------------------------------------------------
re_corrupt = re.compile(
    r"company(rt|nect|unsel|ntend|nsider|nclud|nvict|mmission|mpensat|mplaint|mpetent|ntract|nstitut|nfirm|ntrar)",
    re.I
)

def clean_legal_text(txt):
    if not isinstance(txt, str):
        txt = " ".join(str(x) for x in txt)
    return re_corrupt.sub(r"co\1", txt)

def slice_4_char_zones(txt):
    """
    Slices each long legal case into 4 structural sections prior to tokenization:
      - Zone 0 (Head): First 1,400 chars
      - Zone 1 (Mid) : 45% mark, 1,400 chars
      - Zone 2 (Tail-Ratio): 74%-87% mark, 1,600 chars
      - Zone 3 (Tail-Holding): Final 1,800 chars
    """
    txt = clean_legal_text(txt).strip()
    L = len(txt)
    if L <= 1800:
        return [txt, txt, txt, txt]
    z0 = txt[:1400]
    m_start = int(0.45 * L)
    z1 = txt[m_start : m_start + 1400]
    t2_start = max(0, int(0.74 * L))
    t2_end   = max(t2_start + 400, min(L, t2_start + 1600))
    z2 = txt[t2_start : t2_end]
    z3 = txt[max(0, L - 1800):]
    return [z0, z1, z2, z3]


# ------------------------------------------------------------------------------
# STEP 3: Arch2 Checkpoint Loader & 4-Zone Multi-Layer Feature Extractor
# ------------------------------------------------------------------------------
CACHE_ARCH2 = BASE_OUT / "arch2_4zone_cjpe_features.npz"
N_ZONES = 4
MAX_TOK_LEN = 192

ARCH2_CKPT_PATH = Path("/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt")
if not ARCH2_CKPT_PATH.exists():
    ARCH2_CKPT_PATH = Path("/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/latest.pt")
if not ARCH2_CKPT_PATH.exists():
    found = list(Path("/kaggle/input").rglob("best.pt")) or list(Path("/kaggle/input").rglob("latest.pt"))
    if not found:
        raise FileNotFoundError("Could not locate Arch2 checkpoint (best.pt / latest.pt) under /kaggle/input")
    ARCH2_CKPT_PATH = found[0]

if CACHE_ARCH2.exists():
    print(f"[Step 3] Loading cached Arch2 4-Zone CJPE features from: {CACHE_ARCH2}")
    c_npz = np.load(CACHE_ARCH2)
    X_train_z = c_npz["X_train_zones"].astype(np.float32)
    X_dev_z   = c_npz["X_dev_zones"].astype(np.float32)
    X_test_z  = c_npz["X_test_zones"].astype(np.float32)
else:
    print(f"[Step 3] Loading Arch2 Checkpoint: {ARCH2_CKPT_PATH}")
    ckpt_a2 = torch.load(ARCH2_CKPT_PATH, map_location="cpu", weights_only=False)
    sd_a2   = ckpt_a2.get("model", ckpt_a2.get("state_dict", ckpt_a2))
    sd_a2   = {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in sd_a2.items()}

    tok_path = Path("/kaggle/input/achiver7/phase1-tokens-v40k-clean1/tokens/tokenizer.json")
    if not tok_path.exists():
        tok_path = next(Path("/kaggle/input").rglob("tokenizer.json"))
    tokenizer = Tokenizer.from_file(str(tok_path))
    tokenizer.enable_truncation(max_length=MAX_TOK_LEN - 2)

    VOCAB_SIZE, D_ARCH2 = sd_a2["token_embedding.weight"].shape
    layer_indices = sorted(list({int(m.group(1)) for k in sd_a2.keys() if (m := re.search(r"encoder\.layers\.(\d+)\.", k))}))
    N_LAYERS_A2 = len(layer_indices)
    N_HEADS_A2  = 8
    HEAD_DIM_A2 = D_ARCH2 // N_HEADS_A2
    print(f"  Arch2 Specs -> D_MODEL: {D_ARCH2} | Layers: {N_LAYERS_A2} | Heads: {N_HEADS_A2}")

    sd_gpu = {k: v.to(DEVICE, dtype=torch.float16) for k, v in sd_a2.items() if isinstance(v, torch.Tensor)}

    def get_norm_tensors(prefix_or_name):
        w_key = next((k for k in [f"{prefix_or_name}.weight", f"{prefix_or_name}weight", f"{prefix_or_name}.gamma"] if k in sd_gpu), None)
        b_key = next((k for k in [f"{prefix_or_name}.bias",   f"{prefix_or_name}bias",   f"{prefix_or_name}.beta"]  if k in sd_gpu), None)
        return sd_gpu.get(w_key), sd_gpu.get(b_key)

    # One-time diagnostic: confirm the LayerNorm keys resolve before the big loop
    _l0 = layer_indices[0]
    _w0, _b0 = get_norm_tensors(f"encoder.layers.{_l0}.norm1")
    if _w0 is None:
        _norm_keys = [k for k in sd_gpu if "norm" in k.lower()][:30]
        raise RuntimeError(
            f"Could not locate LayerNorm weight for encoder.layers.{_l0}.norm1. "
            f"Available 'norm' keys in checkpoint: {_norm_keys}"
        )

    @torch.no_grad()
    def forward_arch2(input_ids):
        B, T = input_ids.shape
        x = F.embedding(input_ids, sd_gpu["token_embedding.weight"])

        mid_1 = max(0, N_LAYERS_A2 // 2 - 1)
        mid_2 = max(0, N_LAYERS_A2 - 2)
        h_m1, h_m2 = None, None

        for l_idx in layer_indices:
            p = f"encoder.layers.{l_idx}."
            # FIX: was f"{p[:-1]}norm1" -> produced "encoder.layers.0norm1" (missing dot)
            w1, b1 = get_norm_tensors(f"{p}norm1")
            if w1 is not None:
                h = F.layer_norm(x, (D_ARCH2,), w1, b1, eps=1e-5)
            else:
                h = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-5)

            q = F.linear(h, sd_gpu[f"{p}self_attn.q_proj.weight"], sd_gpu[f"{p}self_attn.q_proj.bias"])
            k = F.linear(h, sd_gpu[f"{p}self_attn.k_proj.weight"], sd_gpu[f"{p}self_attn.k_proj.bias"])
            v = F.linear(h, sd_gpu[f"{p}self_attn.v_proj.weight"], sd_gpu[f"{p}self_attn.v_proj.bias"])

            q = q.view(B, T, N_HEADS_A2, HEAD_DIM_A2).transpose(1, 2)
            k = k.view(B, T, N_HEADS_A2, HEAD_DIM_A2).transpose(1, 2)
            v = v.view(B, T, N_HEADS_A2, HEAD_DIM_A2).transpose(1, 2)

            attn_out = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=True)
            attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, D_ARCH2)
            x = x + F.linear(attn_out, sd_gpu[f"{p}self_attn.out_proj.weight"], sd_gpu[f"{p}self_attn.out_proj.bias"])

            # FIX: was f"{p[:-1]}norm2" -> "encoder.layers.0norm2"
            w2, b2 = get_norm_tensors(f"{p}norm2")
            if w2 is not None:
                h2 = F.layer_norm(x, (D_ARCH2,), w2, b2, eps=1e-5)
            else:
                h2 = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-5)

            ff1 = F.gelu(F.linear(h2, sd_gpu[f"{p}ffn.linear1.weight"], sd_gpu[f"{p}ffn.linear1.bias"]))
            ff2 = F.linear(ff1, sd_gpu[f"{p}ffn.linear2.weight"], sd_gpu[f"{p}ffn.linear2.bias"])
            x = x + ff2

            if l_idx == mid_1: h_m1 = x
            if l_idx == mid_2: h_m2 = x

        # Final norm lookup — broadened fallback chain
        fn_w, fn_b = None, None
        for _nm in ("final_norm", "norm_f", "ln_f", "encoder.norm",
                    "final_layer_norm", "final_ln", "norm"):
            fn_w, fn_b = get_norm_tensors(_nm)
            if fn_w is not None:
                break

        if fn_w is not None:
            h_fin = F.layer_norm(x, (D_ARCH2,), fn_w, fn_b, eps=1e-5)
        else:
            h_fin = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-5)

        return h_m1 if h_m1 is not None else h_fin, h_m2 if h_m2 is not None else h_fin, h_fin

    @torch.no_grad()
    def extract_arch2_zones(records, desc="Arch2 Extract", doc_batch=64):
        N = len(records)
        out = np.zeros((N, N_ZONES, D_ARCH2), dtype=np.float16)
        for start in tqdm(range(0, N, doc_batch), desc=desc):
            end = min(start + doc_batch, N)
            batch_strs = []
            for r in records[start:end]:
                batch_strs.extend(slice_4_char_zones(r["text"]))

            encodings = tokenizer.encode_batch(batch_strs)
            max_l = min(MAX_TOK_LEN, max((len(e.ids) + 2 for e in encodings), default=4))
            inp = np.zeros((len(batch_strs), max_l), dtype=np.int64)
            lengths = np.zeros(len(batch_strs), dtype=np.int64)
            for i_e, enc in enumerate(encodings):
                ids = enc.ids[:MAX_TOK_LEN - 2]
                inp[i_e, 0] = 1
                inp[i_e, 1:1 + len(ids)] = ids
                inp[i_e, 1 + len(ids)] = 2
                lengths[i_e] = len(ids) + 2

            inp_t = torch.from_numpy(inp).to(DEVICE)
            with torch.amp.autocast(DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
                hm1, hm2, hfin = forward_arch2(inp_t)
                h_blend = 0.20 * hm1.float() + 0.30 * hm2.float() + 0.50 * hfin.float()

            arange_t = torch.arange(max_l, device=DEVICE)[None, :]
            len_t = torch.from_numpy(lengths).to(DEVICE)[:, None]
            mask_t = (arange_t < len_t).unsqueeze(-1).float()

            mean_pool = (h_blend * mask_t).sum(dim=1) / torch.clamp(len_t.float(), min=1.0)
            rows = torch.arange(len(batch_strs), device=DEVICE)
            eos_pool = h_blend[rows, len_t.squeeze(-1) - 1]

            chunk_vecs = (0.50 * eos_pool + 0.50 * mean_pool).half().cpu().numpy()
            out[start:end] = chunk_vecs.reshape(end - start, N_ZONES, D_ARCH2)
        return out.astype(np.float32)

    X_train_z = extract_arch2_zones(train_data, desc="Arch2 4-Zone (Train)")
    X_dev_z   = extract_arch2_zones(dev_data,   desc="Arch2 4-Zone (Dev)")
    X_test_z  = extract_arch2_zones(test_data,  desc="Arch2 4-Zone (Test)")

    np.savez_compressed(CACHE_ARCH2,
                        X_train_zones=X_train_z.astype(np.float16),
                        X_dev_zones=X_dev_z.astype(np.float16),
                        X_test_zones=X_test_z.astype(np.float16))
    del sd_gpu, sd_a2, ckpt_a2
    torch.cuda.empty_cache()

X_full_z = np.concatenate([X_train_z, X_dev_z], axis=0)
D_MODEL  = X_full_z.shape[-1]
print(f"[Step 3 Complete] Arch2 Tensor Ready: Full={X_full_z.shape} | Test={X_test_z.shape}")


# ------------------------------------------------------------------------------
# STEP 4: Leak-Free 5-Fold Out-of-Fold Lexical Meta-Features
# ------------------------------------------------------------------------------
def extract_zone_corpora(records):
    tail_docs, ht_docs, arch_flags = [], [], np.zeros(len(records), dtype=np.float32)
    for i, r in enumerate(records):
        txt = clean_legal_text(r["text"]).strip()
        L = len(txt)
        if txt[:120].islower() or "appellate jurisdiction" in txt[:250].lower() or "appeal number" in txt[:250].lower():
            arch_flags[i] = 1.0
        head_s    = txt[:1200]
        mid_s     = txt[int(0.45 * L) : int(0.45 * L) + 1200] if L > 2400 else txt
        tail_5000 = txt[max(0, L - 5000):]
        tail_2000 = txt[max(0, L - 2000):]
        tail_docs.append(f"{tail_5000} [HOLDING] {tail_2000}")
        ht_docs.append(f"[HEAD] {head_s} [MID] {mid_s} [TAIL] {tail_5000}")
    return tail_docs, ht_docs, arch_flags

print("[Step 4] Building Multi-Zone Legal Corpora & Leak-Free 5-Fold OOF Lexical Features...")
tail_full, ht_full, arch_full = extract_zone_corpora(all_records)
tail_test, ht_test, arch_test = extract_zone_corpora(test_data)

vec_tail = TfidfVectorizer(ngram_range=(1, 3), max_features=110000, sublinear_tf=True, min_df=3)
vec_ht   = TfidfVectorizer(ngram_range=(1, 2), max_features=85000,  sublinear_tf=True, min_df=4)

X_tail_full = vec_tail.fit_transform(tail_full)
X_tail_test = vec_tail.transform(tail_test)
X_ht_full   = vec_ht.fit_transform(ht_full)
X_ht_test   = vec_ht.transform(ht_test)

sample_weights = np.where(arch_full > 0.5, 1.85, 1.0).astype(np.float32)
sample_weights[len(train_data):] = 2.50

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_lex  = np.zeros((len(all_records), 3), dtype=np.float32)
test_lex = np.zeros((len(test_data),   3), dtype=np.float32)

for f_idx, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_full)), y_full)):
    sw_tr = sample_weights[tr_idx]
    m_tail_bal = LogisticRegression(C=0.85, max_iter=400, class_weight="balanced", random_state=SEED + f_idx)
    m_ht_bal   = LogisticRegression(C=0.65, max_iter=400, class_weight="balanced", random_state=SEED + f_idx)
    m_ridge    = RidgeClassifier(alpha=1.4, class_weight="balanced", random_state=SEED + f_idx)

    m_tail_bal.fit(X_tail_full[tr_idx], y_full[tr_idx], sample_weight=sw_tr)
    m_ht_bal.fit(X_ht_full[tr_idx],     y_full[tr_idx], sample_weight=sw_tr)
    m_ridge.fit(X_tail_full[tr_idx],    y_full[tr_idx], sample_weight=sw_tr)

    oof_lex[va_idx, 0] = m_tail_bal.predict_proba(X_tail_full[va_idx])[:, 1]
    oof_lex[va_idx, 1] = m_ht_bal.predict_proba(X_ht_full[va_idx])[:, 1]
    oof_lex[va_idx, 2] = expit(m_ridge.decision_function(X_tail_full[va_idx]))

    test_lex[:, 0] += m_tail_bal.predict_proba(X_tail_test)[:, 1] / 5.0
    test_lex[:, 1] += m_ht_bal.predict_proba(X_ht_test)[:, 1] / 5.0
    test_lex[:, 2] += expit(m_ridge.decision_function(X_tail_test)) / 5.0

p_oof_lex_blend  = 0.50 * oof_lex[:, 0]  + 0.30 * oof_lex[:, 1]  + 0.20 * oof_lex[:, 2]
p_test_lex_blend = 0.50 * test_lex[:, 0] + 0.30 * test_lex[:, 1] + 0.20 * test_lex[:, 2]


# ------------------------------------------------------------------------------
# STEP 5: Section-Aware Legal Stance & Cross-Zone Trajectory Features
# ------------------------------------------------------------------------------
re_pos = re.compile(
    r"\b(merit in|substantial force|find force|deserves to succeed|cannot be sustained|"
    r"set aside|quashed|liable to be set aside|entitled to|erroneous|perverse|"
    r"unsustainable|succeeds|in favour of|benefit of doubt|acquitted)\b", re.I
)
re_neg = re.compile(
    r"\b(no merit|devoid of merit|sans merit|deserves to be dismissed|no infirmity|"
    r"no illegality|no perversity|no reason to interfere|decline to interfere|"
    r"concurrent findings|rightly held|well reasoned|warrants no interference|bereft of merit)\b", re.I
)
re_al  = re.compile(r"\b(allow|allowed|allowing|partly allowed|set aside|quash|quashed|modify|modified|remand|remanded)\b", re.I)
re_dm  = re.compile(r"\b(dismiss|dismissed|dismissing|reject|rejected|affirm|affirmed|upheld|confirmed)\b", re.I)

def build_aux_features(records, X_z, lex_cols, arch_f):
    N = len(records)
    feats = np.zeros((N, 18), dtype=np.float32)
    for i, r in enumerate(records):
        txt = clean_legal_text(r["text"]).strip()
        L = len(txt)
        t_4k = txt[max(0, L - 4000):]
        t_1k = txt[max(0, L - 1500):]
        p_4, n_4 = len(re_pos.findall(t_4k)), len(re_neg.findall(t_4k))
        p_1, n_1 = len(re_pos.findall(t_1k)), len(re_neg.findall(t_1k))
        al, dm   = len(re_al.findall(t_1k)),  len(re_dm.findall(t_1k))

        feats[i, 0]  = arch_f[i]
        feats[i, 1]  = np.log1p(L) / 12.0
        feats[i, 2]  = np.log1p(p_4)
        feats[i, 3]  = np.log1p(n_4)
        feats[i, 4]  = (p_4 - n_4) / (p_4 + n_4 + 1.0)
        feats[i, 5]  = np.log1p(p_1)
        feats[i, 6]  = np.log1p(n_1)
        feats[i, 7]  = (p_1 - n_1) / (p_1 + n_1 + 1.0)
        feats[i, 8]  = np.log1p(al)
        feats[i, 9]  = np.log1p(dm)
        feats[i, 10] = (al - dm) / (al + dm + 1.0)

    norm_z = X_z / (np.linalg.norm(X_z, axis=-1, keepdims=True) + 1e-8)
    feats[:, 11] = np.sum(norm_z[:, 0] * norm_z[:, 3], axis=-1)
    feats[:, 12] = np.sum(norm_z[:, 1] * norm_z[:, 3], axis=-1)
    feats[:, 13] = np.sum(norm_z[:, 2] * norm_z[:, 3], axis=-1)
    feats[:, 14] = np.linalg.norm(X_z[:, 3] - X_z[:, 0], axis=-1) / 25.0
    feats[:, 15:18] = lex_cols
    return feats

aux_full = build_aux_features(all_records, X_full_z, oof_lex,  arch_full)
aux_test = build_aux_features(test_data,   X_test_z, test_lex, arch_test)

aux_m = aux_full.mean(axis=0, keepdims=True)
aux_s = aux_full.std(axis=0, keepdims=True) + 1e-6
aux_full_n = (aux_full - aux_m) / aux_s
aux_test_n = (aux_test - aux_m) / aux_s
AUX_DIM = aux_full_n.shape[1]


# ------------------------------------------------------------------------------
# STEP 6: Define Experiment 3 Architecture (BitFit + Layer-Wise Unfreezing)
# ------------------------------------------------------------------------------
class BitFitLinear(nn.Module):
    """BitFit: Weights Frozen (requires_grad=False), Biases Trainable."""
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight   = nn.Parameter(torch.empty(out_features, in_features), requires_grad=False)
        nn.init.orthogonal_(self.weight)
        self.in_bias  = nn.Parameter(torch.zeros(in_features),  requires_grad=True)
        self.out_bias = nn.Parameter(torch.zeros(out_features), requires_grad=True)

    def forward(self, x):
        return F.linear(x + self.in_bias, self.weight, self.out_bias)


class BitFitTransformerBlock(nn.Module):
    def __init__(self, d_model=512, n_heads=8, dropout=0.25):
        super().__init__()
        self.norm1  = nn.LayerNorm(d_model)
        self.q_proj = BitFitLinear(d_model, d_model)
        self.k_proj = BitFitLinear(d_model, d_model)
        self.v_proj = BitFitLinear(d_model, d_model)
        self.o_proj = BitFitLinear(d_model, d_model)
        self.n_heads  = n_heads
        self.head_dim = d_model // n_heads

        self.norm2 = nn.LayerNorm(d_model)
        self.ffn1  = BitFitLinear(d_model, d_model * 2)
        self.ffn2  = BitFitLinear(d_model * 2, d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape
        h = self.norm1(x)
        q = self.q_proj(h).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(h).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(h).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        attn = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0)
        attn = attn.transpose(1, 2).contiguous().view(B, T, D)
        x = x + self.drop(self.o_proj(attn))
        x = x + self.drop(self.ffn2(F.gelu(self.ffn1(self.norm2(x)))))
        return x


class UnfrozenTopTransformerBlock(nn.Module):
    """Layer-Wise Partial Unfreezing: Top layer fully trainable."""
    def __init__(self, d_model=512, n_heads=8, dropout=0.25):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model)
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h1 = self.norm1(x)
        a_out, _ = self.attn(h1, h1, h1)
        x = x + self.drop(a_out)
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x


class Arch2BitFitPartialUnfreezeClassifier(nn.Module):
    def __init__(self, d_in, d_model=512, aux_dim=AUX_DIM, dropout=0.25):
        super().__init__()
        self.norm_in     = nn.LayerNorm(d_in)
        self.stem_bitfit = BitFitLinear(d_in, d_model)
        self.zone_bias   = nn.Parameter(torch.zeros(1, 4, d_model), requires_grad=True)

        self.bitfit_layer1      = BitFitTransformerBlock(d_model=d_model, n_heads=8, dropout=dropout)
        self.bitfit_layer2      = BitFitTransformerBlock(d_model=d_model, n_heads=8, dropout=dropout)
        self.unfrozen_top_layer = UnfrozenTopTransformerBlock(d_model=d_model, n_heads=8, dropout=dropout)
        self.norm_top           = nn.LayerNorm(d_model)

        self.register_buffer("zone_prior", torch.tensor([-0.55, -0.10, 0.35, 0.55], dtype=torch.float32))
        self.zone_scorer = nn.Linear(d_model, 1)

        self.aux_head = nn.Sequential(
            nn.LayerNorm(aux_dim),
            nn.Linear(aux_dim, 96),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(96, 64),
            nn.LayerNorm(64),
            nn.GELU()
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model * 3 + 64),
            nn.Dropout(dropout),
            nn.Linear(d_model * 3 + 64, 256),
            nn.GELU(),
            nn.Dropout(dropout * 0.7),
            nn.Linear(256, 2)
        )

    def forward(self, x_zones, x_aux):
        if self.training:
            m0 = (torch.rand(x_zones.size(0), 1, 1, device=x_zones.device) < 0.35).float()
            x_zones = x_zones.clone()
            x_zones[:, 0:1, :] = x_zones[:, 0:1, :] * (1.0 - m0)

        h = F.gelu(self.stem_bitfit(self.norm_in(x_zones))) + self.zone_bias
        h = self.bitfit_layer1(h)
        h = self.bitfit_layer2(h)
        h_top = self.norm_top(self.unfrozen_top_layer(h))

        zw = F.softmax(self.zone_scorer(h_top).squeeze(-1) + self.zone_prior[None, :], dim=-1)
        pooled = torch.sum(h_top * zw.unsqueeze(-1), dim=1)

        feat_cat = torch.cat([pooled, h_top[:, 3], h_top[:, 3] - h_top[:, 0], self.aux_head(x_aux)], dim=-1)
        return self.classifier(feat_cat)


# ------------------------------------------------------------------------------
# STEP 7: Train 3-Seed Full-Corpus Ensemble (100% Data = 39,375 Cases)
# ------------------------------------------------------------------------------
print("\n" + "=" * 78)
print("[Step 7] TRAINING EXPERIMENT 3: ARCH2 + BITFIT & TOP-LAYER UNFREEZING (3 SEEDS)")
print("=" * 78)

counts_full = np.bincount(y_full, minlength=2).astype(np.float64)
cls_weights = torch.tensor((len(y_full) / counts_full) ** 0.55, dtype=torch.float32, device=DEVICE)
cls_weights /= cls_weights.mean()

full_ds = torch.utils.data.TensorDataset(
    torch.from_numpy(X_full_z),
    torch.from_numpy(aux_full_n),
    torch.from_numpy(y_full),
    torch.from_numpy(sample_weights)
)
full_loader = torch.utils.data.DataLoader(full_ds, batch_size=256, shuffle=True)

X_te_t = torch.from_numpy(X_test_z).to(DEVICE)
A_te_t = torch.from_numpy(aux_test_n).to(DEVICE)
X_fu_t = torch.from_numpy(X_full_z)
A_fu_t = torch.from_numpy(aux_full_n)

SEEDS = [42, 1024, 2026]
test_nn_probs_seeds  = np.zeros(len(test_data), dtype=np.float32)
full_nn_logits_accum = np.zeros((len(all_records), 2), dtype=np.float32)
test_nn_logits_accum = np.zeros((len(test_data),   2), dtype=np.float32)

for s_i, seed_val in enumerate(SEEDS, 1):
    torch.manual_seed(seed_val)
    np.random.seed(seed_val)

    model = Arch2BitFitPartialUnfreezeClassifier(d_in=D_MODEL, d_model=512, aux_dim=AUX_DIM, dropout=0.25).to(DEVICE)

    bitfit_params = [
        p for n, p in model.named_parameters()
        if p.requires_grad and ("bias" in n or "bitfit" in n or "norm" in n)
    ]
    bitfit_ids = {id(p) for p in bitfit_params}
    top_params = [p for p in model.parameters() if p.requires_grad and id(p) not in bitfit_ids]

    optimizer = torch.optim.AdamW([
        {"params": bitfit_params, "lr": 1.2e-3, "weight_decay": 1e-4},
        {"params": top_params,    "lr": 3.5e-4, "weight_decay": 1.5e-2}
    ])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=9, eta_min=1e-5)

    for ep in range(1, 10):
        model.train()
        total_loss = 0.0
        for bz, ba, by, bsw in full_loader:
            bz, ba, by, bsw = bz.to(DEVICE), ba.to(DEVICE), by.to(DEVICE), bsw.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(bz, ba)
            per_sample_loss = F.cross_entropy(
                logits, by, weight=cls_weights, label_smoothing=0.03, reduction="none"
            )
            loss = (per_sample_loss * bsw).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        if ep in (1, 5, 9):
            print(f"  [Seed {s_i}/{len(SEEDS)}] Epoch {ep}/9 | Domain-Weighted Loss: {total_loss/len(full_loader):.4f}")

    model.eval()
    with torch.no_grad():
        te_l = []
        for start in range(0, len(test_data), 256):
            end = min(start + 256, len(test_data))
            te_l.append(model(X_te_t[start:end], A_te_t[start:end]).cpu().numpy())
        te_l = np.concatenate(te_l, axis=0)
        test_nn_logits_accum += te_l / len(SEEDS)
        test_nn_probs_seeds  += softmax(te_l, axis=-1)[:, 1] / len(SEEDS)

        fu_l = []
        for start in range(0, len(all_records), 512):
            end = min(start + 512, len(all_records))
            fu_l.append(model(X_fu_t[start:end].to(DEVICE), A_fu_t[start:end].to(DEVICE)).cpu().numpy())
        full_nn_logits_accum += np.concatenate(fu_l, axis=0) / len(SEEDS)


# ------------------------------------------------------------------------------
# STEP 8: Refit Full-Corpus Lexical Models, Fuse Probabilities & Export ZIP
# ------------------------------------------------------------------------------
m_tail_full = LogisticRegression(C=0.85, max_iter=400, class_weight="balanced", random_state=SEED)
m_ht_full   = LogisticRegression(C=0.65, max_iter=400, class_weight="balanced", random_state=SEED)
m_rdg_full  = RidgeClassifier(alpha=1.4, class_weight="balanced", random_state=SEED)

m_tail_full.fit(X_tail_full, y_full, sample_weight=sample_weights)
m_ht_full.fit(X_ht_full,     y_full, sample_weight=sample_weights)
m_rdg_full.fit(X_tail_full,  y_full, sample_weight=sample_weights)

p_test_lex_full = (
    0.50 * m_tail_full.predict_proba(X_tail_test)[:, 1] +
    0.30 * m_ht_full.predict_proba(X_ht_test)[:, 1] +
    0.20 * expit(m_rdg_full.decision_function(X_tail_test))
)

p_test_final = 0.75 * test_nn_probs_seeds + 0.125 * p_test_lex_blend + 0.125 * p_test_lex_full

# Save Experiment 3 Teacher Artifacts for Experiment 4 Knowledge Distillation
np.savez_compressed(
    EXP3_DIR / "exp3_arch2_bitfit_teacher_probs.npz",
    full_logits=full_nn_logits_accum,
    test_logits=test_nn_logits_accum,
    p_test_final=p_test_final,
    p_oof_lex_blend=p_oof_lex_blend
)

median_thr = float(np.percentile(p_test_final, 50.0))
test_preds = (p_test_final >= median_thr).astype(np.int64)

submission_list = [
    {"id": str(r["id"]), "pred": int(p)}
    for r, p in zip(test_data, test_preds)
]

assert isinstance(submission_list, list) and len(submission_list) == len(test_data)
for item, orig in zip(submission_list, test_data):
    assert item["id"] == str(orig["id"])
    assert isinstance(item["pred"], int) and item["pred"] in (0, 1)

pred_json_path = EXP3_DIR / "prediction.json"
sub_zip_path   = EXP3_DIR / "submission.zip"

with open(pred_json_path, "w", encoding="utf-8") as f:
    json.dump(submission_list, f, ensure_ascii=False, indent=2)
with zipfile.ZipFile(sub_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_json_path, arcname="prediction.json")

print("=" * 78)
print("EXPERIMENT 3 (ARCH2 + BITFIT & LAYER-WISE UNFREEZING) SUBMISSION READY")
print("=" * 78)
print(f"Balanced 50/50 Threshold : {median_thr:.4f}")
print("Test Prediction Counts   :", Counter(x["pred"] for x in submission_list))
print("First 3 predictions      :", submission_list[:3])
print("Submission ZIP saved at  :", sub_zip_path)

[Step 1] Full Training Corpus: 39375 cases | Label Counts: Counter({np.int64(0): 23061, np.int64(1): 16314})
[Step 3] Loading Arch2 Checkpoint: /kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt
  Arch2 Specs -> D_MODEL: 768 | Layers: 6 | Heads: 8


Arch2 4-Zone (Train):   0%|          | 0/585 [00:00<?, ?it/s]

Arch2 4-Zone (Dev):   0%|          | 0/32 [00:00<?, ?it/s]

Arch2 4-Zone (Test):   0%|          | 0/24 [00:00<?, ?it/s]

[Step 3 Complete] Arch2 Tensor Ready: Full=(39375, 4, 768) | Test=(1517, 4, 768)
[Step 4] Building Multi-Zone Legal Corpora & Leak-Free 5-Fold OOF Lexical Features...

[Step 7] TRAINING EXPERIMENT 3: ARCH2 + BITFIT & TOP-LAYER UNFREEZING (3 SEEDS)
  [Seed 1/3] Epoch 1/9 | Domain-Weighted Loss: 0.6443
  [Seed 1/3] Epoch 5/9 | Domain-Weighted Loss: 0.5541
  [Seed 1/3] Epoch 9/9 | Domain-Weighted Loss: 0.5448
  [Seed 2/3] Epoch 1/9 | Domain-Weighted Loss: 0.6485
  [Seed 2/3] Epoch 5/9 | Domain-Weighted Loss: 0.5553
  [Seed 2/3] Epoch 9/9 | Domain-Weighted Loss: 0.5445
  [Seed 3/3] Epoch 1/9 | Domain-Weighted Loss: 0.6419
  [Seed 3/3] Epoch 5/9 | Domain-Weighted Loss: 0.5526
  [Seed 3/3] Epoch 9/9 | Domain-Weighted Loss: 0.5447
EXPERIMENT 3 (ARCH2 + BITFIT & LAYER-WISE UNFREEZING) SUBMISSION READY
Balanced 50/50 Threshold : 0.4717
Test Prediction Counts   : Counter({1: 759, 0: 758})
First 3 predictions      : [{'id': '1986_397', 'pred': 0}, {'id': '1977_183', 'pred': 0}, {'id': '1993_98', 

In [5]:
# ==============================================================================
# PHASE 4 (CJPE) — EXPERIMENT 3 (v2): IMPROVED ARCH2 + BITFIT + PARTIAL UNFREEZE
# ==============================================================================
# Key changes vs. v1:
#   - 6 zones (Head, Early-Mid, Mid, Late-Mid, Pre-Tail, Tail) instead of 4
#   - MAX_TOK_LEN 256 (was 192)
#   - d_model = 768 (matches Arch2 output); frozen 768->512 stem projection REMOVED
#   - BitFit on Layer 1 only; Layers 2 & 3 fully trainable (layer-wise unfreezing)
#   - Attention pooling over zones (learned query + positional prior)
#   - 12 epochs with 2-epoch linear warmup + cosine decay
#   - 5 seeds instead of 3
#   - TruncatedSVD(64) of TF-IDF appended to aux features
#   - 15% holdout pass to tune fusion weights + threshold; then retrain on 100%
#   - Temperature scaling on NN probabilities before fusion
# ==============================================================================

import os, re, json, math, random, zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import Counter
from tokenizers import Tokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score
from scipy.special import expit, softmax, logit
from tqdm.auto import tqdm

# ------------------------------------------------------------------------------
# STEP 0: Repro & Device
# ------------------------------------------------------------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_ROOT = Path("/kaggle/working/Phase_4_CJPE/Phase_4_CJPE")
BASE_OUT  = Path("/kaggle/working/Phase4_Submissions")
EXP3_DIR  = BASE_OUT / "Exp3_FullTrain_Arch2_BitFit_Unfreeze_v2"
EXP3_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# STEP 1: Data
# ------------------------------------------------------------------------------
with open(DATA_ROOT / "Train/train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(DATA_ROOT / "Train/dev.json", "r", encoding="utf-8") as f:
    dev_data = json.load(f)
with open(DATA_ROOT / "Test/test.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

all_records = train_data + dev_data
y_full = np.array([int(x["label"]) for x in all_records], dtype=np.int64)
print(f"[Step 1] Full corpus: {len(all_records)} | Labels: {Counter(y_full)}")

# ------------------------------------------------------------------------------
# STEP 2: De-corruption & 6-Zone slicer
# ------------------------------------------------------------------------------
re_corrupt = re.compile(
    r"company(rt|nect|unsel|ntend|nsider|nclud|nvict|mmission|mpensat|mplaint|mpetent|ntract|nstitut|nfirm|ntrar)",
    re.I
)

def clean_legal_text(txt):
    if not isinstance(txt, str):
        txt = " ".join(str(x) for x in txt)
    return re_corrupt.sub(r"co\1", txt)

def slice_6_char_zones(txt):
    """6-zone slicing: Head / Early-Mid / Mid / Late-Mid / Pre-Tail / Tail-Holding."""
    txt = clean_legal_text(txt).strip()
    L = len(txt)
    if L <= 1800:
        return [txt] * 6
    z0 = txt[:1400]
    z1 = txt[int(0.20 * L): int(0.20 * L) + 1400]
    z2 = txt[int(0.40 * L): int(0.40 * L) + 1400]
    z3 = txt[int(0.60 * L): int(0.60 * L) + 1400]
    z4_s = max(0, int(0.76 * L))
    z4_e = max(z4_s + 400, min(L, z4_s + 1600))
    z4 = txt[z4_s:z4_e]
    z5 = txt[max(0, L - 1800):]
    return [z0, z1, z2, z3, z4, z5]

# ------------------------------------------------------------------------------
# STEP 3: Arch2 6-Zone Feature Extraction (with corrected LayerNorm keys)
# ------------------------------------------------------------------------------
N_ZONES = 6
MAX_TOK_LEN = 256
CACHE_ARCH2 = BASE_OUT / "arch2_6zone_cjpe_features_v2.npz"

ARCH2_CKPT_PATH = Path("/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt")
if not ARCH2_CKPT_PATH.exists():
    ARCH2_CKPT_PATH = Path("/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/latest.pt")
if not ARCH2_CKPT_PATH.exists():
    found = list(Path("/kaggle/input").rglob("best.pt")) or list(Path("/kaggle/input").rglob("latest.pt"))
    if not found:
        raise FileNotFoundError("Arch2 checkpoint not found")
    ARCH2_CKPT_PATH = found[0]

if CACHE_ARCH2.exists():
    print(f"[Step 3] Loading cache: {CACHE_ARCH2}")
    c_npz = np.load(CACHE_ARCH2)
    X_train_z = c_npz["X_train_zones"].astype(np.float32)
    X_dev_z   = c_npz["X_dev_zones"].astype(np.float32)
    X_test_z  = c_npz["X_test_zones"].astype(np.float32)
else:
    print(f"[Step 3] Extracting from: {ARCH2_CKPT_PATH}")
    ckpt = torch.load(ARCH2_CKPT_PATH, map_location="cpu", weights_only=False)
    sd = ckpt.get("model", ckpt.get("state_dict", ckpt))
    sd = {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in sd.items()}

    tok_path = Path("/kaggle/input/achiver7/phase1-tokens-v40k-clean1/tokens/tokenizer.json")
    if not tok_path.exists():
        tok_path = next(Path("/kaggle/input").rglob("tokenizer.json"))
    tokenizer = Tokenizer.from_file(str(tok_path))
    tokenizer.enable_truncation(max_length=MAX_TOK_LEN - 2)

    VOCAB_SIZE, D_ARCH2 = sd["token_embedding.weight"].shape
    layer_indices = sorted({int(m.group(1)) for k in sd if (m := re.search(r"encoder\.layers\.(\d+)\.", k))})
    N_LAYERS_A2 = len(layer_indices)
    N_HEADS_A2  = 8
    HEAD_DIM_A2 = D_ARCH2 // N_HEADS_A2
    print(f"  Arch2: D={D_ARCH2}, L={N_LAYERS_A2}, H={N_HEADS_A2}")

    sd_gpu = {k: v.to(DEVICE, dtype=torch.float16) for k, v in sd.items() if isinstance(v, torch.Tensor)}

    def get_norm_tensors(name):
        w_key = next((k for k in (f"{name}.weight", f"{name}weight", f"{name}.gamma") if k in sd_gpu), None)
        b_key = next((k for k in (f"{name}.bias",   f"{name}bias",   f"{name}.beta")  if k in sd_gpu), None)
        return sd_gpu.get(w_key), sd_gpu.get(b_key)

    @torch.no_grad()
    def forward_arch2(input_ids):
        B, T = input_ids.shape
        x = F.embedding(input_ids, sd_gpu["token_embedding.weight"])
        mid_1 = max(0, N_LAYERS_A2 // 2 - 1)
        mid_2 = max(0, N_LAYERS_A2 - 2)
        h_m1 = h_m2 = None

        for l in layer_indices:
            p = f"encoder.layers.{l}."
            w1, b1 = get_norm_tensors(f"{p}norm1")
            h = F.layer_norm(x, (D_ARCH2,), w1, b1, eps=1e-5) if w1 is not None else x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-5)

            q = F.linear(h, sd_gpu[f"{p}self_attn.q_proj.weight"], sd_gpu[f"{p}self_attn.q_proj.bias"])
            k = F.linear(h, sd_gpu[f"{p}self_attn.k_proj.weight"], sd_gpu[f"{p}self_attn.k_proj.bias"])
            v = F.linear(h, sd_gpu[f"{p}self_attn.v_proj.weight"], sd_gpu[f"{p}self_attn.v_proj.bias"])
            q = q.view(B, T, N_HEADS_A2, HEAD_DIM_A2).transpose(1, 2)
            k = k.view(B, T, N_HEADS_A2, HEAD_DIM_A2).transpose(1, 2)
            v = v.view(B, T, N_HEADS_A2, HEAD_DIM_A2).transpose(1, 2)
            a = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=True)
            a = a.transpose(1, 2).contiguous().view(B, T, D_ARCH2)
            x = x + F.linear(a, sd_gpu[f"{p}self_attn.out_proj.weight"], sd_gpu[f"{p}self_attn.out_proj.bias"])

            w2, b2 = get_norm_tensors(f"{p}norm2")
            h2 = F.layer_norm(x, (D_ARCH2,), w2, b2, eps=1e-5) if w2 is not None else x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-5)
            ff = F.linear(F.gelu(F.linear(h2, sd_gpu[f"{p}ffn.linear1.weight"], sd_gpu[f"{p}ffn.linear1.bias"])),
                          sd_gpu[f"{p}ffn.linear2.weight"], sd_gpu[f"{p}ffn.linear2.bias"])
            x = x + ff
            if l == mid_1: h_m1 = x
            if l == mid_2: h_m2 = x

        fn_w = fn_b = None
        for nm in ("final_norm", "norm_f", "ln_f", "encoder.norm", "final_layer_norm", "final_ln", "norm"):
            fn_w, fn_b = get_norm_tensors(nm)
            if fn_w is not None:
                break
        h_fin = F.layer_norm(x, (D_ARCH2,), fn_w, fn_b, eps=1e-5) if fn_w is not None else x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-5)
        return (h_m1 if h_m1 is not None else h_fin,
                h_m2 if h_m2 is not None else h_fin,
                h_fin)

    @torch.no_grad()
    def extract_zones(records, desc, doc_batch=64):
        N = len(records)
        out = np.zeros((N, N_ZONES, D_ARCH2), dtype=np.float16)
        for start in tqdm(range(0, N, doc_batch), desc=desc):
            end = min(start + doc_batch, N)
            batch_strs = []
            for r in records[start:end]:
                batch_strs.extend(slice_6_char_zones(r["text"]))
            encs = tokenizer.encode_batch(batch_strs)
            max_l = min(MAX_TOK_LEN, max((len(e.ids) + 2 for e in encs), default=4))
            inp = np.zeros((len(batch_strs), max_l), dtype=np.int64)
            lens = np.zeros(len(batch_strs), dtype=np.int64)
            for i_e, e in enumerate(encs):
                ids = e.ids[:MAX_TOK_LEN - 2]
                inp[i_e, 0] = 1
                inp[i_e, 1:1 + len(ids)] = ids
                inp[i_e, 1 + len(ids)] = 2
                lens[i_e] = len(ids) + 2

            inp_t = torch.from_numpy(inp).to(DEVICE)
            with torch.amp.autocast(DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
                hm1, hm2, hfin = forward_arch2(inp_t)
                h_blend = 0.20 * hm1.float() + 0.30 * hm2.float() + 0.50 * hfin.float()

            arange = torch.arange(max_l, device=DEVICE)[None, :]
            len_t = torch.from_numpy(lens).to(DEVICE)[:, None]
            mask = (arange < len_t).unsqueeze(-1).float()
            mean_pool = (h_blend * mask).sum(1) / torch.clamp(len_t.float(), min=1.0)
            rows = torch.arange(len(batch_strs), device=DEVICE)
            eos_pool = h_blend[rows, len_t.squeeze(-1) - 1]
            chunk_vecs = (0.5 * eos_pool + 0.5 * mean_pool).half().cpu().numpy()
            out[start:end] = chunk_vecs.reshape(end - start, N_ZONES, D_ARCH2)
        return out.astype(np.float32)

    X_train_z = extract_zones(train_data, "Arch2 (Train)")
    X_dev_z   = extract_zones(dev_data,   "Arch2 (Dev)")
    X_test_z  = extract_zones(test_data,  "Arch2 (Test)")

    np.savez_compressed(CACHE_ARCH2,
                        X_train_zones=X_train_z.astype(np.float16),
                        X_dev_zones=X_dev_z.astype(np.float16),
                        X_test_zones=X_test_z.astype(np.float16))
    del sd_gpu, sd, ckpt
    torch.cuda.empty_cache()

X_full_z = np.concatenate([X_train_z, X_dev_z], axis=0)
D_MODEL = X_full_z.shape[-1]
print(f"[Step 3] Full={X_full_z.shape} | Test={X_test_z.shape}")

# ------------------------------------------------------------------------------
# STEP 4: TF-IDF corpora + 5-fold OOF lexical features + SVD
# ------------------------------------------------------------------------------
def extract_zone_corpora(records):
    tail_docs, ht_docs, arch_flags = [], [], np.zeros(len(records), dtype=np.float32)
    for i, r in enumerate(records):
        txt = clean_legal_text(r["text"]).strip(); L = len(txt)
        if txt[:120].islower() or "appellate jurisdiction" in txt[:250].lower() or "appeal number" in txt[:250].lower():
            arch_flags[i] = 1.0
        head_s = txt[:1200]
        mid_s = txt[int(0.45 * L): int(0.45 * L) + 1200] if L > 2400 else txt
        tail_5000 = txt[max(0, L - 5000):]
        tail_2000 = txt[max(0, L - 2000):]
        tail_docs.append(f"{tail_5000} [HOLDING] {tail_2000}")
        ht_docs.append(f"[HEAD] {head_s} [MID] {mid_s} [TAIL] {tail_5000}")
    return tail_docs, ht_docs, arch_flags

print("[Step 4] TF-IDF + OOF lex + SVD...")
tail_full, ht_full, arch_full = extract_zone_corpora(all_records)
tail_test, ht_test, arch_test = extract_zone_corpora(test_data)

vec_tail = TfidfVectorizer(ngram_range=(1, 3), max_features=110000, sublinear_tf=True, min_df=3)
vec_ht   = TfidfVectorizer(ngram_range=(1, 2), max_features=85000,  sublinear_tf=True, min_df=4)
X_tail_full = vec_tail.fit_transform(tail_full); X_tail_test = vec_tail.transform(tail_test)
X_ht_full   = vec_ht.fit_transform(ht_full);     X_ht_test   = vec_ht.transform(ht_test)

# SVD(64) of tail TF-IDF -> aux features
svd = TruncatedSVD(n_components=64, random_state=SEED)
svd_full = svd.fit_transform(X_tail_full).astype(np.float32)
svd_test = svd.transform(X_tail_test).astype(np.float32)

sample_weights = np.where(arch_full > 0.5, 1.85, 1.0).astype(np.float32)
sample_weights[len(train_data):] = 2.50

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_lex  = np.zeros((len(all_records), 3), dtype=np.float32)
test_lex = np.zeros((len(test_data),   3), dtype=np.float32)

for f_idx, (tr, va) in enumerate(skf.split(np.zeros(len(y_full)), y_full)):
    sw = sample_weights[tr]
    m1 = LogisticRegression(C=0.85, max_iter=400, class_weight="balanced", random_state=SEED + f_idx)
    m2 = LogisticRegression(C=0.65, max_iter=400, class_weight="balanced", random_state=SEED + f_idx)
    m3 = RidgeClassifier(alpha=1.4, class_weight="balanced", random_state=SEED + f_idx)
    m1.fit(X_tail_full[tr], y_full[tr], sample_weight=sw)
    m2.fit(X_ht_full[tr],   y_full[tr], sample_weight=sw)
    m3.fit(X_tail_full[tr], y_full[tr], sample_weight=sw)
    oof_lex[va, 0] = m1.predict_proba(X_tail_full[va])[:, 1]
    oof_lex[va, 1] = m2.predict_proba(X_ht_full[va])[:, 1]
    oof_lex[va, 2] = expit(m3.decision_function(X_tail_full[va]))
    test_lex[:, 0] += m1.predict_proba(X_tail_test)[:, 1] / 5.0
    test_lex[:, 1] += m2.predict_proba(X_ht_test)[:, 1] / 5.0
    test_lex[:, 2] += expit(m3.decision_function(X_tail_test)) / 5.0

p_oof_lex_blend  = 0.50 * oof_lex[:, 0]  + 0.30 * oof_lex[:, 1]  + 0.20 * oof_lex[:, 2]
p_test_lex_blend = 0.50 * test_lex[:, 0] + 0.30 * test_lex[:, 1] + 0.20 * test_lex[:, 2]

# ------------------------------------------------------------------------------
# STEP 5: Aux features (regex + cross-zone similarity + SVD)
# ------------------------------------------------------------------------------
re_pos = re.compile(
    r"\b(merit in|substantial force|find force|deserves to succeed|cannot be sustained|"
    r"set aside|quashed|liable to be set aside|entitled to|erroneous|perverse|unsustainable|"
    r"succeeds|in favour of|benefit of doubt|acquitted|clear error|manifestly unjust)\b", re.I
)
re_neg = re.compile(
    r"\b(no merit|devoid of merit|sans merit|deserves to be dismissed|no infirmity|no illegality|"
    r"no perversity|no reason to interfere|decline to interfere|concurrent findings|rightly held|"
    r"well reasoned|warrants no interference|bereft of merit|no substance|no force)\b", re.I
)
re_al = re.compile(r"\b(allow|allowed|allowing|partly allowed|set aside|quash|quashed|modify|modified|remand|remanded)\b", re.I)
re_dm = re.compile(r"\b(dismiss|dismissed|dismissing|reject|rejected|affirm|affirmed|upheld|confirmed)\b", re.I)

def build_aux(records, X_z, lex_cols, svd_feats, arch_f):
    N = len(records)
    feats = np.zeros((N, 18), dtype=np.float32)
    for i, r in enumerate(records):
        txt = clean_legal_text(r["text"]).strip(); L = len(txt)
        t_4k = txt[max(0, L - 4000):]; t_1k = txt[max(0, L - 1500):]
        p4, n4 = len(re_pos.findall(t_4k)), len(re_neg.findall(t_4k))
        p1, n1 = len(re_pos.findall(t_1k)), len(re_neg.findall(t_1k))
        al, dm = len(re_al.findall(t_1k)), len(re_dm.findall(t_1k))
        feats[i, 0] = arch_f[i]
        feats[i, 1] = np.log1p(L) / 12.0
        feats[i, 2] = np.log1p(p4); feats[i, 3] = np.log1p(n4)
        feats[i, 4] = (p4 - n4) / (p4 + n4 + 1.0)
        feats[i, 5] = np.log1p(p1); feats[i, 6] = np.log1p(n1)
        feats[i, 7] = (p1 - n1) / (p1 + n1 + 1.0)
        feats[i, 8] = np.log1p(al); feats[i, 9] = np.log1p(dm)
        feats[i, 10] = (al - dm) / (al + dm + 1.0)
    nz = X_z / (np.linalg.norm(X_z, axis=-1, keepdims=True) + 1e-8)
    feats[:, 11] = np.sum(nz[:, 0] * nz[:, 5], axis=-1)   # head<->tail
    feats[:, 12] = np.sum(nz[:, 2] * nz[:, 5], axis=-1)   # mid<->tail
    feats[:, 13] = np.sum(nz[:, 4] * nz[:, 5], axis=-1)   # pre-tail<->tail
    feats[:, 14] = np.linalg.norm(X_z[:, 5] - X_z[:, 0], axis=-1) / 25.0
    feats[:, 15:18] = lex_cols
    return np.concatenate([feats, svd_feats], axis=-1)

aux_full = build_aux(all_records, X_full_z, oof_lex, svd_full, arch_full)
aux_test = build_aux(test_data,   X_test_z, test_lex, svd_test, arch_test)
aux_m = aux_full.mean(0, keepdims=True); aux_s = aux_full.std(0, keepdims=True) + 1e-6
aux_full_n = ((aux_full - aux_m) / aux_s).astype(np.float32)
aux_test_n = ((aux_test - aux_m) / aux_s).astype(np.float32)
AUX_DIM = aux_full_n.shape[1]
print(f"[Step 5] Aux dim = {AUX_DIM}")

# ------------------------------------------------------------------------------
# STEP 6: Model — BitFit L1, fully-trainable L2 & L3, attention pooling
# ------------------------------------------------------------------------------
class BitFitLinear(nn.Module):
    def __init__(self, in_f, out_f):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_f, in_f), requires_grad=False)
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        self.in_bias  = nn.Parameter(torch.zeros(in_f))
        self.out_bias = nn.Parameter(torch.zeros(out_f))
    def forward(self, x):
        return F.linear(x + self.in_bias, self.weight, self.out_bias)

class BitFitBlock(nn.Module):
    """BitFit: frozen linear weights, trainable biases + LayerNorm affine."""
    def __init__(self, d_model, n_heads=8, dropout=0.2):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.q = BitFitLinear(d_model, d_model)
        self.k = BitFitLinear(d_model, d_model)
        self.v = BitFitLinear(d_model, d_model)
        self.o = BitFitLinear(d_model, d_model)
        self.n_heads = n_heads; self.head_dim = d_model // n_heads
        self.norm2 = nn.LayerNorm(d_model)
        self.ff1 = BitFitLinear(d_model, d_model * 2)
        self.ff2 = BitFitLinear(d_model * 2, d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        B, T, D = x.shape
        h = self.norm1(x)
        q = self.q(h).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k(h).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v(h).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        a = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0)
        a = a.transpose(1, 2).contiguous().view(B, T, D)
        x = x + self.drop(self.o(a))
        x = x + self.drop(self.ff2(F.gelu(self.ff1(self.norm2(x)))))
        return x

class UnfrozenBlock(nn.Module):
    """Layer-wise unfreezing: fully trainable transformer block."""
    def __init__(self, d_model, n_heads=8, dropout=0.2):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
        )
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        h = self.norm1(x)
        a, _ = self.attn(h, h, h)
        x = x + self.drop(a)
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x

class Arch2BitFitV2(nn.Module):
    def __init__(self, d_model=768, aux_dim=AUX_DIM, n_zones=6, dropout=0.2):
        super().__init__()
        self.norm_in = nn.LayerNorm(d_model)
        self.zone_bias = nn.Parameter(torch.zeros(1, n_zones, d_model))
        self.bitfit_l1 = BitFitBlock(d_model, n_heads=8, dropout=dropout)
        self.unfrozen_l2 = UnfrozenBlock(d_model, n_heads=8, dropout=dropout)
        self.unfrozen_l3 = UnfrozenBlock(d_model, n_heads=8, dropout=dropout)
        self.norm_top = nn.LayerNorm(d_model)
        # attention pooling
        self.zone_query = nn.Parameter(torch.randn(d_model) * 0.02)
        self.register_buffer("zone_prior",
                             torch.tensor([-0.40, -0.20, -0.10, 0.10, 0.30, 0.50], dtype=torch.float32))
        # heads
        self.aux_head = nn.Sequential(
            nn.LayerNorm(aux_dim), nn.Linear(aux_dim, 128), nn.GELU(),
            nn.Dropout(dropout * 0.5), nn.Linear(128, 64), nn.LayerNorm(64), nn.GELU(),
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model * 3 + 64), nn.Dropout(dropout),
            nn.Linear(d_model * 3 + 64, 384), nn.GELU(), nn.Dropout(dropout * 0.7),
            nn.Linear(384, 2),
        )
    def forward(self, x_zones, x_aux):
        if self.training:
            # 30% head dropout per example
            m0 = (torch.rand(x_zones.size(0), 1, 1, device=x_zones.device) < 0.30).float()
            x_zones = x_zones.clone()
            x_zones[:, 0:1, :] = x_zones[:, 0:1, :] * (1.0 - m0)
        h = self.norm_in(x_zones) + self.zone_bias
        h = self.bitfit_l1(h)
        h = self.unfrozen_l2(h)
        h = self.unfrozen_l3(h)
        h_top = self.norm_top(h)
        # attention pooling over zones
        scores = torch.einsum("bnd,d->bn", h_top, self.zone_query) + self.zone_prior[None, :]
        zw = F.softmax(scores, dim=-1)
        pooled = torch.sum(h_top * zw.unsqueeze(-1), dim=1)
        tail = h_top[:, -1]
        head = h_top[:, 0]
        return self.classifier(torch.cat(
            [pooled, tail, tail - head, self.aux_head(x_aux)], dim=-1
        ))

# ------------------------------------------------------------------------------
# STEP 7: Training helpers
# ------------------------------------------------------------------------------
counts = np.bincount(y_full, minlength=2).astype(np.float64)
cls_weights = torch.tensor((len(y_full) / counts) ** 0.55, dtype=torch.float32, device=DEVICE)
cls_weights /= cls_weights.mean()

def train_one(tr_idx, ep_total=12, warmup=2, batch=256, lr_bitfit=1.2e-3, lr_top=3.5e-4,
              seed=42, X=None, A=None, Y=None, SW=None, verbose=False):
    torch.manual_seed(seed); np.random.seed(seed)
    model = Arch2BitFitV2(d_model=D_MODEL, aux_dim=AUX_DIM, dropout=0.20).to(DEVICE)

    bitfit_params = [p for n, p in model.named_parameters()
                     if p.requires_grad and ("bias" in n or "bitfit" in n or "norm" in n)]
    bitfit_ids = {id(p) for p in bitfit_params}
    top_params = [p for p in model.parameters() if p.requires_grad and id(p) not in bitfit_ids]

    opt = torch.optim.AdamW([
        {"params": bitfit_params, "lr": lr_bitfit, "weight_decay": 1e-4},
        {"params": top_params,    "lr": lr_top,    "weight_decay": 1.5e-2},
    ])
    # linear warmup then cosine
    def lr_lambda(ep):
        if ep < warmup:
            return (ep + 1) / max(1, warmup)
        p = (ep - warmup) / max(1, ep_total - warmup)
        return 0.5 * (1 + math.cos(math.pi * p))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    ds = torch.utils.data.TensorDataset(
        torch.from_numpy(X[tr_idx]),
        torch.from_numpy(A[tr_idx]),
        torch.from_numpy(Y[tr_idx]),
        torch.from_numpy(SW[tr_idx]),
    )
    loader = torch.utils.data.DataLoader(ds, batch_size=batch, shuffle=True)

    for ep in range(ep_total):
        model.train(); total = 0.0
        for bz, ba, by, bsw in loader:
            bz, ba, by, bsw = bz.to(DEVICE), ba.to(DEVICE), by.to(DEVICE), bsw.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = (F.cross_entropy(model(bz, ba), by, weight=cls_weights,
                                    label_smoothing=0.05, reduction="none") * bsw).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item()
        sched.step()
        if verbose and (ep in (0, ep_total // 2, ep_total - 1)):
            print(f"    ep {ep+1}/{ep_total} | loss {total/len(loader):.4f}")
    model.eval()
    return model

@torch.no_grad()
def predict(model, X, A, batch=512):
    model.eval()
    out = []
    for s in range(0, len(X), batch):
        e = min(s + batch, len(X))
        out.append(model(torch.from_numpy(X[s:e]).to(DEVICE),
                         torch.from_numpy(A[s:e]).to(DEVICE)).cpu())
    return torch.cat(out).numpy()

def sigmoid(x): return 1.0 / (1.0 + np.exp(-x))

def fit_temperature(logits, y, grid=np.arange(0.5, 3.01, 0.05)):
    best_T, best_nll = 1.0, 1e18
    for T in grid:
        p = sigmoid(logits / T)
        nll = -np.mean(y * np.log(p + 1e-9) + (1 - y) * np.log(1 - p + 1e-9))
        if nll < best_nll:
            best_nll, best_T = nll, T
    return best_T

# ------------------------------------------------------------------------------
# STEP 8: 15% HOLD-OUT PASS — tune fusion weights, temperature, threshold
# ------------------------------------------------------------------------------
print("\n" + "=" * 78)
print("[Step 8] HOLD-OUT PASS (15%) for fusion / temperature / threshold tuning")
print("=" * 78)

idx = np.arange(len(all_records))
tr_idx, va_idx = train_test_split(idx, test_size=0.15, stratify=y_full, random_state=SEED)
print(f"  Train: {len(tr_idx)} | Val: {len(va_idx)}")

holdout_logits = np.zeros((len(va_idx), 2), dtype=np.float32)
for s in range(2):  # 2 seeds averaged for stable OOF estimate
    m = train_one(tr_idx, seed=SEED + s * 1000,
                  X=X_full_z, A=aux_full_n, Y=y_full, SW=sample_weights, verbose=False)
    holdout_logits += predict(m, X_full_z[va_idx], aux_full_n[va_idx]) / 2.0

# Convert to probability of positive class
oof_nn_p = softmax(holdout_logits, axis=-1)[:, 1].astype(np.float32)
oof_nn_logit = logit(np.clip(oof_nn_p, 1e-6, 1 - 1e-6))

# Temperature calibration on holdout
T_best = fit_temperature(oof_nn_logit, y_full[va_idx])
oof_nn_p_cal = sigmoid(oof_nn_logit / T_best)
print(f"  Calibrated temperature T = {T_best:.2f}")

# Optimize fusion weights (2-way: NN vs lex blend) on holdout Macro-F1
best_f1, best_w, best_thr = -1.0, None, None
for w_nn in np.arange(0.4, 1.001, 0.025):
    w_lex = 1.0 - w_nn
    p_val = w_nn * oof_nn_p_cal + w_lex * p_oof_lex_blend[va_idx]
    for thr in np.arange(0.30, 0.71, 0.01):
        f1 = f1_score(y_full[va_idx], (p_val >= thr).astype(int), average="macro")
        if f1 > best_f1:
            best_f1, best_w, best_thr = f1, (w_nn, w_lex), thr

print(f"  Tuned fusion on holdout: NN={best_w[0]:.2f}  Lex={best_w[1]:.2f}")
print(f"  Tuned threshold        : {best_thr:.3f}")
print(f"  Holdout Macro-F1       : {best_f1:.4f}")

# For reference: baseline (0.75 / 0.125 / 0.125 style fusion, median threshold)
p_val_base = 0.75 * oof_nn_p + 0.25 * p_oof_lex_blend[va_idx]
thr_base = np.median(p_val_base)
f1_base = f1_score(y_full[va_idx], (p_val_base >= thr_base).astype(int), average="macro")
print(f"  Baseline holdout F1    : {f1_base:.4f}  (gain from tuning: {best_f1 - f1_base:+.4f})")

# ------------------------------------------------------------------------------
# STEP 9: FULL PASS — train N seeds on 100% data
# ------------------------------------------------------------------------------
print("\n" + "=" * 78)
print("[Step 9] FULL PASS — training on 100% data")
print("=" * 78)

FULL_SEEDS = [42, 1024, 2026, 3140, 5077]
test_logits_accum = np.zeros((len(test_data), 2), dtype=np.float32)
full_logits_accum = np.zeros((len(all_records), 2), dtype=np.float32)
all_idx = np.arange(len(all_records))

for si, sv in enumerate(FULL_SEEDS, 1):
    print(f"  Seed {si}/{len(FULL_SEEDS)} ({sv})")
    m = train_one(all_idx, seed=sv,
                  X=X_full_z, A=aux_full_n, Y=y_full, SW=sample_weights, verbose=False)
    test_logits_accum += predict(m, X_test_z, aux_test_n) / len(FULL_SEEDS)
    full_logits_accum += predict(m, X_full_z, aux_full_n) / len(FULL_SEEDS)

# Apply the calibration + fusion + threshold learned on the holdout
test_nn_p = softmax(test_logits_accum, axis=-1)[:, 1]
test_nn_logit = logit(np.clip(test_nn_p, 1e-6, 1 - 1e-6))
test_nn_p_cal = sigmoid(test_nn_logit / T_best)

p_test_final = best_w[0] * test_nn_p_cal + best_w[1] * p_test_lex_blend
test_preds = (p_test_final >= best_thr).astype(np.int64)

# ------------------------------------------------------------------------------
# STEP 10: Export
# ------------------------------------------------------------------------------
submission_list = [{"id": str(r["id"]), "pred": int(p)} for r, p in zip(test_data, test_preds)]
assert len(submission_list) == len(test_data)
for it, orig in zip(submission_list, test_data):
    assert it["id"] == str(orig["id"]) and it["pred"] in (0, 1)

np.savez_compressed(
    EXP3_DIR / "exp3_v2_teacher_probs.npz",
    full_logits=full_logits_accum, test_logits=test_logits_accum,
    p_test_final=p_test_final, p_oof_lex_blend=p_oof_lex_blend,
    temperature=T_best, fusion_w=np.array(best_w, dtype=np.float32), threshold=best_thr,
)

pred_json_path = EXP3_DIR / "prediction.json"
sub_zip_path   = EXP3_DIR / "submission.zip"
with open(pred_json_path, "w", encoding="utf-8") as f:
    json.dump(submission_list, f, ensure_ascii=False, indent=2)
with zipfile.ZipFile(sub_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_json_path, arcname="prediction.json")

print("=" * 78)
print("EXP3 v2 READY")
print("=" * 78)
print(f"Temperature         : {T_best:.3f}")
print(f"Fusion              : NN={best_w[0]:.3f}  Lex={best_w[1]:.3f}")
print(f"Threshold           : {best_thr:.3f}")
print(f"Test preds          : {Counter(x['pred'] for x in submission_list)}")
print(f"ZIP                 : {sub_zip_path}")

[Step 1] Full corpus: 39375 | Labels: Counter({np.int64(0): 23061, np.int64(1): 16314})
[Step 3] Extracting from: /kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt
  Arch2: D=768, L=6, H=8


Arch2 (Train):   0%|          | 0/585 [00:00<?, ?it/s]

Arch2 (Dev):   0%|          | 0/32 [00:00<?, ?it/s]

Arch2 (Test):   0%|          | 0/24 [00:00<?, ?it/s]

[Step 3] Full=(39375, 6, 768) | Test=(1517, 6, 768)
[Step 4] TF-IDF + OOF lex + SVD...
[Step 5] Aux dim = 82

[Step 8] HOLD-OUT PASS (15%) for fusion / temperature / threshold tuning
  Train: 33468 | Val: 5907
  Calibrated temperature T = 1.00
  Tuned fusion on holdout: NN=0.40  Lex=0.60
  Tuned threshold        : 0.500
  Holdout Macro-F1       : 0.7647
  Baseline holdout F1    : 0.7594  (gain from tuning: +0.0053)

[Step 9] FULL PASS — training on 100% data
  Seed 1/5 (42)
  Seed 2/5 (1024)
  Seed 3/5 (2026)
  Seed 4/5 (3140)
  Seed 5/5 (5077)
EXP3 v2 READY
Temperature         : 1.000
Fusion              : NN=0.400  Lex=0.600
Threshold           : 0.500
Test preds          : Counter({0: 774, 1: 743})
ZIP                 : /kaggle/working/Phase4_Submissions/Exp3_FullTrain_Arch2_BitFit_Unfreeze_v2/submission.zip


In [6]:
# ==============================================================================
# EXPERIMENT 3 — FULL FINE-TUNE Legal-BERT-small (no LoRA — avoids torchao bug)
# + Stacked ensemble with existing lexical OOF
# ==============================================================================
# Changes vs. prior attempt:
#   - SKIP LoRA entirely (Legal-BERT-small fits for full FT on T4)
#   - lr = 2e-5 (correct BERT FT range; 3e-4 was a bug)
#   - Per-fold checkpointing (resume-safe)
#   - Fold-level progress saved to disk
#   - Reports OOF-vs-test gap explicitly
# ==============================================================================

import os, re, json, math, random, zipfile, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.isotonic import IsotonicRegression
from scipy.special import expit
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

QUICK_MODE = False   # True = 1 fold, 15% data, 1 epoch, ~6 min sanity check

DATA_ROOT = Path("/kaggle/working/Phase_4_CJPE/Phase_4_CJPE")
BASE_OUT  = Path("/kaggle/working/Phase4_Submissions")
FT_DIR    = BASE_OUT / "Exp3_LegalBERT_FullFT_Stack"
FT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------------------------
with open(DATA_ROOT / "Train/train.json") as f: train_data = json.load(f)
with open(DATA_ROOT / "Train/dev.json")   as f: dev_data   = json.load(f)
with open(DATA_ROOT / "Test/test.json")   as f: test_data  = json.load(f)

all_records = train_data + dev_data
y_full = np.array([int(x["label"]) for x in all_records], dtype=np.int64)
N_FULL, N_TEST = len(all_records), len(test_data)
print(f"[1] Full={N_FULL}  Test={N_TEST}  Labels={Counter(y_full)}")

re_corrupt = re.compile(
    r"company(rt|nect|unsel|ntend|nsider|nclud|nvict|mmission|mpensat|mplaint|mpetent|ntract|nstitut|nfirm|ntrar)",
    re.I)
def clean_legal_text(txt):
    if not isinstance(txt, str): txt = " ".join(str(x) for x in txt)
    return re_corrupt.sub(r"co\1", txt)

def prepare_text(txt, max_chars=2000):
    """Head + tail — 1000 chars each. ~512 tokens for a BERT-family encoder."""
    txt = clean_legal_text(txt).strip()
    if len(txt) <= max_chars:
        return txt
    half = max_chars // 2
    return txt[:half] + " ... " + txt[-half:]

texts_full = [prepare_text(r["text"]) for r in all_records]
texts_test = [prepare_text(r["text"]) for r in test_data]

arch_full = np.zeros(N_FULL, dtype=np.float32)
for i, r in enumerate(all_records):
    t = clean_legal_text(r["text"]).strip()[:250].lower()
    if t[:120].islower() or "appellate jurisdiction" in t or "appeal number" in t:
        arch_full[i] = 1.0
sample_weights = np.where(arch_full > 0.5, 1.85, 1.0).astype(np.float32)
sample_weights[len(train_data):] = 2.50

# ------------------------------------------------------------------------------
# 2. Lexical OOF (5-fold) — cached
# ------------------------------------------------------------------------------
LEX_CACHE = FT_DIR / "lex_oof.npz"
if LEX_CACHE.exists():
    print("\n[2] Loading cached lexical OOF...")
    lc = np.load(LEX_CACHE)
    oof_lex  = lc["oof_lex"]
    test_lex = lc["test_lex"]
else:
    print("\n[2] Computing lexical OOF (5-fold)...")
    def zone_corpora(records):
        tails, hts = [], []
        for r in records:
            txt = clean_legal_text(r["text"]).strip(); L = len(txt)
            h = txt[:1200]
            m = txt[int(0.45 * L):int(0.45 * L) + 1200] if L > 2400 else txt
            t5 = txt[max(0, L - 5000):]; t2 = txt[max(0, L - 2000):]
            tails.append(f"{t5} [HOLDING] {t2}")
            hts.append(f"[HEAD] {h} [MID] {m} [TAIL] {t5}")
        return tails, hts

    tail_full, ht_full = zone_corpora(all_records)
    tail_test, ht_test = zone_corpora(test_data)

    vec_tail = TfidfVectorizer(ngram_range=(1, 3), max_features=110000, sublinear_tf=True, min_df=3)
    vec_ht   = TfidfVectorizer(ngram_range=(1, 2), max_features=85000,  sublinear_tf=True, min_df=4)
    X_tail_full = vec_tail.fit_transform(tail_full); X_tail_test = vec_tail.transform(tail_test)
    X_ht_full   = vec_ht.fit_transform(ht_full);     X_ht_test   = vec_ht.transform(ht_test)

    skf_lex = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof_lex  = np.zeros((N_FULL, 3), dtype=np.float32)
    test_lex = np.zeros((N_TEST, 3), dtype=np.float32)
    for f_idx, (tr, va) in enumerate(skf_lex.split(np.zeros(N_FULL), y_full)):
        sw = sample_weights[tr]
        m1 = LogisticRegression(C=0.85, max_iter=400, class_weight="balanced", random_state=SEED + f_idx)
        m2 = LogisticRegression(C=0.65, max_iter=400, class_weight="balanced", random_state=SEED + f_idx)
        m3 = RidgeClassifier(alpha=1.4, class_weight="balanced", random_state=SEED + f_idx)
        m1.fit(X_tail_full[tr], y_full[tr], sample_weight=sw)
        m2.fit(X_ht_full[tr],   y_full[tr], sample_weight=sw)
        m3.fit(X_tail_full[tr], y_full[tr], sample_weight=sw)
        oof_lex[va, 0] = m1.predict_proba(X_tail_full[va])[:, 1]
        oof_lex[va, 1] = m2.predict_proba(X_ht_full[va])[:, 1]
        oof_lex[va, 2] = expit(m3.decision_function(X_tail_full[va]))
        test_lex[:, 0] += m1.predict_proba(X_tail_test)[:, 1] / 5.0
        test_lex[:, 1] += m2.predict_proba(X_ht_test)[:, 1] / 5.0
        test_lex[:, 2] += expit(m3.decision_function(X_tail_test)) / 5.0
    np.savez_compressed(LEX_CACHE, oof_lex=oof_lex, test_lex=test_lex)

for k, nm in enumerate(["Lex tail LR", "Lex HMT LR", "Lex tail Ridge"]):
    print(f"  {nm}: OOF F1={f1_score(y_full, (oof_lex[:, k] >= 0.5).astype(int), average='macro'):.4f}")

# ------------------------------------------------------------------------------
# 3. Load encoder (NO LoRA — avoids torchao dispatch)
# ------------------------------------------------------------------------------
print("\n[3] Loading encoder (full FT, no LoRA)...")
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_CANDIDATES = [
    "nlpaueb/legal-bert-small-uncased",
    "microsoft/deberta-v3-small",
    "distilbert-base-uncased",
]
MODEL_NAME, tokenizer = None, None
for name in MODEL_CANDIDATES:
    try:
        tokenizer = AutoTokenizer.from_pretrained(name)
        _ = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2, ignore_mismatched_sizes=True)
        MODEL_NAME = name; break
    except Exception as e:
        print(f"  {name}: {e}")
        continue
if MODEL_NAME is None:
    raise RuntimeError("Could not load any encoder. Enable internet in Kaggle settings.")

# Clean up the temporary model from the candidate test
del _
torch.cuda.empty_cache()
print(f"  Using: {MODEL_NAME}")

# ------------------------------------------------------------------------------
# 4. Fine-tuning helpers (FULL FINE-TUNE — no LoRA)
# ------------------------------------------------------------------------------
class TextDataset(Dataset):
    def __init__(self, texts, labels, weights):
        self.texts, self.labels, self.weights = texts, labels, weights
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        return {"text": self.texts[i], "label": int(self.labels[i]), "weight": float(self.weights[i])}

class WeightedCollator:
    def __init__(self, tok, max_len=512):
        self.tok = tok; self.max_len = max_len
    def __call__(self, batch):
        enc = self.tok([b["text"] for b in batch], truncation=True,
                       max_length=self.max_len, padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor([b["label"] for b in batch], dtype=torch.long)
        enc["weights"] = torch.tensor([b["weight"] for b in batch], dtype=torch.float32)
        return enc

def train_encoder(tr_idx, seed=42, epochs=3, lr=2e-5, bs=16):
    """Full fine-tune. lr=2e-5 is standard BERT FT range (2e-5 to 5e-5)."""
    torch.manual_seed(seed); np.random.seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True).to(DEVICE)

    # BERT FT: lower LR for encoder, higher for classifier head
    no_decay = ["bias", "LayerNorm.weight"]
    encoder_params = [
        {"params": [p for n, p in model.named_parameters()
                    if "classifier" not in n and not any(nd in n for nd in no_decay)],
         "weight_decay": 0.01, "lr": lr},
        {"params": [p for n, p in model.named_parameters()
                    if "classifier" not in n and any(nd in n for nd in no_decay)],
         "weight_decay": 0.0, "lr": lr},
        {"params": [p for n, p in model.named_parameters() if "classifier" in n],
         "weight_decay": 0.0, "lr": lr * 10},   # head at 10x LR
    ]
    opt = torch.optim.AdamW(encoder_params)

    steps_per_ep = max(1, len(tr_idx) // bs)
    total_steps = steps_per_ep * epochs
    warmup_steps = int(0.1 * total_steps)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s / max(1, warmup_steps) if s < warmup_steps
        else max(0.0, (total_steps - s) / max(1, total_steps - warmup_steps)))

    ds = TextDataset([texts_full[i] for i in tr_idx],
                     [y_full[i] for i in tr_idx],
                     [sample_weights[i] for i in tr_idx])
    loader = DataLoader(ds, batch_size=bs, shuffle=True,
                        collate_fn=WeightedCollator(tokenizer), num_workers=2)

    model.train()
    for ep in range(epochs):
        pbar = tqdm(loader, desc=f"    ep {ep+1}/{epochs}", leave=False)
        for batch in pbar:
            ids = batch["input_ids"].to(DEVICE)
            attn = batch["attention_mask"].to(DEVICE)
            lbl = batch["labels"].to(DEVICE)
            w = batch["weights"].to(DEVICE)
            logits = model(input_ids=ids, attention_mask=attn).logits
            loss = (F.cross_entropy(logits, lbl, label_smoothing=0.05, reduction="none") * w).mean()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            pbar.set_postfix(loss=f"{loss.item():.3f}")
    return model

@torch.no_grad()
def predict_encoder(model, texts, bs=64):
    model.eval()
    out = []
    for s in range(0, len(texts), bs):
        e = min(s + bs, len(texts))
        enc = tokenizer(texts[s:e], truncation=True, max_length=512,
                        padding=True, return_tensors="pt").to(DEVICE)
        logits = model(**enc).logits
        out.append(F.softmax(logits, dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(out)

# ------------------------------------------------------------------------------
# 5. 5-fold OOF with per-fold checkpointing (resume-safe)
# ------------------------------------------------------------------------------
print("\n[5] Fine-tuning encoder (5-fold OOF)...")
N_FOLDS = 1 if QUICK_MODE else 5
N_EPOCHS = 1 if QUICK_MODE else 3

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(skf.split(np.zeros(N_FULL), y_full))[:N_FOLDS]
if QUICK_MODE:
    folds = [(tr[np.random.choice(len(tr), size=len(tr)//7, replace=False)], va) for tr, va in folds]

oof_enc  = np.zeros(N_FULL, dtype=np.float32)
test_enc = np.zeros(N_TEST, dtype=np.float32)
n_folds_done = 0

for fold, (tr, va) in enumerate(folds):
    fold_ckpt = FT_DIR / f"fold_{fold}_oof.npz"
    if fold_ckpt.exists():
        print(f"  Fold {fold+1}/{N_FOLDS}: loading cached predictions")
        fk = np.load(fold_ckpt)
        oof_enc[va] = fk["oof"]
        test_enc += fk["test"]
        n_folds_done += 1
        f1 = f1_score(y_full[va], (fk["oof"] >= 0.5).astype(int), average="macro")
        print(f"    Val F1@0.5 = {f1:.4f}")
        continue

    print(f"  Fold {fold+1}/{N_FOLDS}: train={len(tr)} val={len(va)}")
    model = train_encoder(tr, seed=SEED + fold * 7919, epochs=N_EPOCHS)
    oof_fold = predict_encoder(model, [texts_full[i] for i in va])
    test_fold = predict_encoder(model, texts_test)
    oof_enc[va] = oof_fold
    test_enc += test_fold
    n_folds_done += 1
    np.savez_compressed(fold_ckpt, oof=oof_fold, test=test_fold)
    f1 = f1_score(y_full[va], (oof_fold >= 0.5).astype(int), average="macro")
    print(f"    Val F1@0.5 = {f1:.4f}")
    del model; torch.cuda.empty_cache()

test_enc /= max(1, n_folds_done)

enc_oof_f1 = f1_score(y_full, (oof_enc >= 0.5).astype(int), average="macro")
print(f"\n  Encoder OOF F1@0.5 = {enc_oof_f1:.4f}")

np.savez_compressed(FT_DIR / "encoder_oof.npz",
                    oof_enc=oof_enc, test_enc=test_enc, model_name=np.array([MODEL_NAME]))

# ------------------------------------------------------------------------------
# 6. Stacking meta-learner
# ------------------------------------------------------------------------------
print("\n[6] Stacking...")

BASE_FULL = np.stack([oof_enc, oof_lex[:, 0], oof_lex[:, 1], oof_lex[:, 2]], axis=1)
BASE_TEST = np.stack([test_enc, test_lex[:, 0], test_lex[:, 1], test_lex[:, 2]], axis=1)

def to_rank(p):
    order = np.argsort(np.argsort(p))
    return order.astype(np.float32) / (len(p) - 1)
BASE_FULL_R = np.stack([to_rank(BASE_FULL[:, k]) for k in range(4)], axis=1)
BASE_TEST_R = np.stack([to_rank(BASE_TEST[:, k]) for k in range(4)], axis=1)

META_FULL = np.concatenate([BASE_FULL, BASE_FULL_R], axis=1)
META_TEST = np.concatenate([BASE_TEST, BASE_TEST_R], axis=1)

skf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 999)
oof_meta = np.zeros(N_FULL, dtype=np.float32)
test_meta = np.zeros(N_TEST, dtype=np.float32)
for f_idx, (tr, va) in enumerate(skf_meta.split(np.zeros(N_FULL), y_full)):
    lr = LogisticRegression(C=0.5, max_iter=600, class_weight="balanced",
                            random_state=SEED + f_idx)
    lr.fit(META_FULL[tr], y_full[tr])
    oof_meta[va] = lr.predict_proba(META_FULL[va])[:, 1]
    test_meta += lr.predict_proba(META_TEST)[:, 1] / 5.0

meta_f1 = f1_score(y_full, (oof_meta >= 0.5).astype(int), average="macro")
print(f"  Meta OOF F1@0.5 = {meta_f1:.4f}")

# ------------------------------------------------------------------------------
# 7. Isotonic + macro-F1 threshold
# ------------------------------------------------------------------------------
print("\n[7] Calibration + threshold...")
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(oof_meta, y_full)
oof_cal = iso.predict(oof_meta).astype(np.float32)
test_cal = iso.predict(test_meta).astype(np.float32)

ths = np.arange(0.30, 0.71, 0.002)
f1s = [f1_score(y_full, (oof_cal >= t).astype(int), average="macro") for t in ths]
best_idx = int(np.argmax(f1s))
best_thr = float(ths[best_idx])
best_f1 = float(f1s[best_idx])
print(f"  Best threshold = {best_thr:.3f}")
print(f"  Calibrated OOF Macro-F1 = {best_f1:.4f}")

# ------------------------------------------------------------------------------
# 8. Export
# ------------------------------------------------------------------------------
print("\n[8] Exporting...")
test_preds = (test_cal >= best_thr).astype(np.int64)
submission = [{"id": str(r["id"]), "pred": int(p)} for r, p in zip(test_data, test_preds)]
assert len(submission) == len(test_data)
for it, orig in zip(submission, test_data):
    assert it["id"] == str(orig["id"]) and it["pred"] in (0, 1)

np.savez_compressed(FT_DIR / "final_probs.npz",
                    oof_base=BASE_FULL, test_base=BASE_TEST,
                    oof_meta=oof_meta, test_meta=test_meta,
                    oof_cal=oof_cal, test_cal=test_cal,
                    best_thr=np.float32(best_thr), oof_f1=np.float32(best_f1))

pred_json = FT_DIR / "prediction.json"
sub_zip   = FT_DIR / "submission.zip"
with open(pred_json, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)
with zipfile.ZipFile(sub_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_json, arcname="prediction.json")

print("\n" + "=" * 78)
print("DONE")
print("=" * 78)
print(f"Encoder             : {MODEL_NAME}")
print(f"Encoder OOF F1      : {enc_oof_f1:.4f}")
print(f"Meta OOF F1         : {meta_f1:.4f}")
print(f"Calibrated OOF F1   : {best_f1:.4f}  (thr={best_thr:.3f})")
print(f"Test preds          : {Counter(x['pred'] for x in submission)}")
print(f"ZIP                 : {sub_zip}")
print()
print("NOTE on OOF vs test gap:")
print("  Your lexical models showed OOF F1 ≈ 0.75 but v1 test score was 0.7053.")
print("  Expect test Macro-F1 to be ~0.02–0.05 lower than the OOF number above.")

[1] Full=39375  Test=1517  Labels=Counter({np.int64(0): 23061, np.int64(1): 16314})

[2] Computing lexical OOF (5-fold)...
  Lex tail LR: OOF F1=0.7511
  Lex HMT LR: OOF F1=0.7303
  Lex tail Ridge: OOF F1=0.7591

[3] Loading encoder (full FT, no LoRA)...


config.json:   0%|          | 0.00/989 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  141MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

... (32 outputs left out)

    ep 2/3:   0%|          | 0/1969 [00:00<?, ?it/s]

    ep 3/3:   0%|          | 0/1969 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e6e2dd93380>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e6e2dd93380>  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers

    Traceback (most recent call last):
if w.is_alive():  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__

  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
        self._shutdown_workers()assert self._parent_pid == os.getpid(), 'can only test a child process'

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
AssertionError    : if w.is_alive():can only test a child process

  File "/usr/lib/

    Val F1@0.5 = 0.7415

  Encoder OOF F1@0.5 = 0.7370

[6] Stacking...
  Meta OOF F1@0.5 = 0.7733

[7] Calibration + threshold...
  Best threshold = 0.462
  Calibrated OOF Macro-F1 = 0.7774

[8] Exporting...

DONE
Encoder             : nlpaueb/legal-bert-small-uncased
Encoder OOF F1      : 0.7370
Meta OOF F1         : 0.7733
Calibrated OOF F1   : 0.7774  (thr=0.462)
Test preds          : Counter({0: 953, 1: 564})
ZIP                 : /kaggle/working/Phase4_Submissions/Exp3_LegalBERT_FullFT_Stack/submission.zip

NOTE on OOF vs test gap:
  Your lexical models showed OOF F1 ≈ 0.75 but v1 test score was 0.7053.
  Expect test Macro-F1 to be ~0.02–0.05 lower than the OOF number above.
